In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# === 数据准备（复用上节） ===
data = [
    ("这个 电影 太 好看 了", 1), ("演员 演技 在线 剧情 精彩", 1),
    ("非常 喜欢 这部 作品", 1), ("画面 精美 值得 一看", 1),
    ("太 浪费 时间 了 烂片", 0), ("剧情 拖沓 演技 尴尬", 0),
    ("完全 看 不 下去", 0), ("浪费 钱 不 推荐", 0),
    ("这 片子 真 不错", 1), ("故事 感人 强烈 推荐", 1),
    ("烂 到 极点 别看", 0), ("毫无 亮点 失望", 0),
]
tokenized = [(t.split(), l) for t, l in data]
counter = {}
for tokens, _ in tokenized:
    for t in tokens:
        counter[t] = counter.get(t, 0) + 1
word2idx = {"<pad>": 0, "<unk>": 1}
for word in sorted(counter, key=counter.get, reverse=True):
    word2idx[word] = len(word2idx)


class SentimentDataset(Dataset):
    def __init__(self, data, w2i):
        self.data = data
        self.w2i = w2i

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]
        return torch.tensor([self.w2i.get(t, 1) for t in tokens]), torch.tensor(label)


def collate_fn(batch):
    texts, labels = zip(*batch)
    return pad_sequence(texts, batch_first=True), torch.stack(labels)


loader = DataLoader(SentimentDataset(tokenized, word2idx), batch_size=4, shuffle=True, collate_fn=collate_fn)

In [7]:
for texts, labels in loader:
    print(f"文本形状: {texts.shape}")
    print(f"标签: {labels}")
    break

文本形状: torch.Size([4, 5])
标签: tensor([0, 0, 1, 1])


In [8]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (h_n, _) = self.lstm(embedded)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)

In [9]:
model = LSTMClassifier(vocab_size=len(word2idx), embed_dim=32, hidden_dim=64, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [10]:
for epoch in range(30):
    total_loss, correct, total = 0, 0, 0

    for texts, labels in loader:
        logits = model(texts)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:2d} | Loss: {total_loss:.4f} | Acc: {correct / total:.2%}")

Epoch 10 | Loss: 1.7098 | Acc: 100.00%
Epoch 20 | Loss: 0.0599 | Acc: 100.00%
Epoch 30 | Loss: 0.0075 | Acc: 100.00%


In [11]:
torch.save(model.state_dict(), "lstm.pth")
print("模型权重保存成功")

模型权重保存成功


In [12]:
torch.save(model, "lstm_full_model.pt")

In [13]:
import pickle
with open("word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)